# Week 1 Assignment — E-Commerce Dataset Cleaning

**Objective:** Inspect, clean, validate, and document an e-commerce dataset while preserving the original source file.

**Execution environment:** Google Colab  
**Input:** `/content/drive/MyDrive/e-commerce_dataset.xlsx`  
**Output folder:** `/content/drive/MyDrive/data-science-week1-assignment/`

> **Data integrity principle:** The raw workbook is read only. Cleaning is conservative: only confirmed fully blank rows and exact duplicate records are removed automatically. Missing transaction fields, suspicious prices/quantities, repeated order IDs, category variants, and total mismatches are reported for review rather than silently corrected.

## 1. Setup and mount Google Drive

Run the notebook from top to bottom. If the filename or Drive location differs, update `file_path`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
from IPython.display import display

file_path = "/content/drive/MyDrive/e-commerce_dataset.xlsx"
project_dir = Path("/content/drive/MyDrive/data-science-week1-assignment")
data_cleaned_dir = project_dir / "data" / "cleaned"
outputs_dir = project_dir / "outputs"
reports_dir = project_dir / "reports"
for folder in [data_cleaned_dir, outputs_dir, reports_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if not Path(file_path).exists():
    raise FileNotFoundError(
        f"File not found: {file_path}. Check the filename and location in Google Drive."
    )

print("Raw source:", file_path)
print("Project output folder:", project_dir)


Mounted at /content/drive
Raw source: /content/drive/MyDrive/e-commerce_dataset.xlsx
Project output folder: /content/drive/MyDrive/data-science-week1-assignment


## 2. Load and inspect the raw dataset

The first worksheet is loaded by default. If the workbook has multiple sheets, review sheet names first and set `sheet_name` to the appropriate one.

In [2]:
xls = pd.ExcelFile(file_path, engine="openpyxl")
print("Available worksheets:", xls.sheet_names)

sheet_name = 0
df_raw = pd.read_excel(file_path, sheet_name=sheet_name, engine="openpyxl")
df_loaded = df_raw.copy(deep=True)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nData types:")
display(pd.DataFrame({
    "column": df_raw.columns.astype(str),
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns]
}))
print("\nFirst 10 rows:")
display(df_raw.head(10))
print("\nDataFrame information:")
df_raw.info()


Available worksheets: ['e-Commerce Dataset']
Rows: 10,034
Columns: 16
Memory usage: 7.31 MB

Data types:


,column,dtype
0,Order ID,object
1,Order Date,object
2,Customer ID,object
3,Customer Name,object
4,Product ID,object
5,Product Name,object
6,Category,object
7,Subcategory,object
8,Quantity,float64
9,Unit Price,float64



First 10 rows:


,Order ID,Order Date,Customer ID,Customer Name,Product ID,Product Name,Category,Subcategory,Quantity,Unit Price,Total Amount,Discount,Payment Mode,Delivery Status,City,Region
0,CA-2017-151428,2017-09-21 00:00:00,RH-19495,Rick Hansen,OFF-BI-10000546,Avery Durable Binders,Office Supplies,Binders,7.0,2.88,20.16,0.0,Debit Card,Delivered,Rochester,Central
1,CA-2017-137470,2017-09-17 00:00:00,TP-21415,Tom Prescott,OFF-PA-10002001,Xerox 1984,Office Supplies,Paper,2.0,6.48,12.96,0.0,Credit Card,Delivered,Seattle,West
2,CA-2017-138779,2017-01-14 00:00:00,RB-19570,Rob Beeghly,OFF-EN-10002504,"Tyvek Top-Opening Peel & Seel Envelopes, Plai...",Office Supplies,Envelopes,1.0,21.74,21.74,0.2,Debit Card,Delivered,Jacksonville,South
3,CA-2015-149678,2015-04-13 00:00:00,AW-10840,Anthony Witt,OFF-SU-10004498,Martin-Yale Premier Letter Opener,Office Supplies,Supplies,1.0,12.88,12.88,0.0,Credit Card,Returned,Farmington,West
4,CA-2016-146682,2016-10-29 00:00:00,KW-16435,Katrina Willman,FUR-FU-10002671,Electrix 20W Halogen Replacement Bulb for Zoom...,Furniture,Furnishings,5.0,13.40,67.00,0.0,Debit Card,Processing,Lansing,Central
5,CA-2015-135020,2015-05-28 00:00:00,MS-17770,Maxwell Schwartz,TEC-PH-10004093,Panasonic Kx-TS550,Technology,Phones,1.0,45.99,45.99,0.0,UPI,Delivered,Rochester,East
6,CA-2016-162747,2016-03-20 00:00:00,AH-10030,Aaron Hawkins,FUR-FU-10003691,"Eldon Image Series Desk Accessories, Ebony",Furniture,Furnishings,7.0,NaN,86.45,0.0,Net Banking,Delivered,Gulfport,South
7,CA-2015-118948,2015-05-28 00:00:00,NK-18490,Neil Knudson,OFF-AR-10001547,Newell 311,Furniture,Art,3.0,2.21,6.63,0.0,Cash on Delivery,Delivered,Seattle,West
8,CA-2017-143798,2017-12-10 00:00:00,AW-10840,Anthony Witt,FUR-FU-10004306,Electrix Halogen Magnifier Lamp,Furniture,Furnishings,2.0,155.44,310.88,0.2,Debit Card,Delivered,Philadelphia,East
9,CA-2014-164315,2014-04-02 00:00:00,RD-19585,Rob Dowd,OFF-PA-10004248,Xerox 1990,Office Supplies,Paper,3.0,5.28,15.84,0.0,Cash on Delivery,Processing,Athens,South



DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10034 entries, 0 to 10033
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order ID         10034 non-null  object 
 1   Order Date       9972 non-null   object 
 2   Customer ID      9975 non-null   object 
 3   Customer Name    9915 non-null   object 
 4   Product ID       10034 non-null  object 
 5   Product Name     9955 non-null   object 
 6   Category         9914 non-null   object 
 7   Subcategory      10034 non-null  object 
 8   Quantity         9955 non-null   float64
 9   Unit Price       9913 non-null   float64
 10  Total Amount     10034 non-null  float64
 11  Discount         10034 non-null  float64
 12  Payment Mode     9854 non-null   object 
 13  Delivery Status  9853 non-null   object 
 14  City             9914 non-null   object 
 15  Region           10034 non-null  object 
dtypes: float64(4), object(12)
memory u

## 3. Standardize column names and create a business dictionary

Column names are converted to lowercase `snake_case`. A collision check stops the notebook if two original names would become identical, so no column is silently overwritten. Business meanings are suggestions based on common names and must be confirmed against the dataset documentation.

In [3]:
def standardize_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_") or "unnamed_column"

original_columns = list(df_raw.columns)
standard_columns = [standardize_column_name(c) for c in original_columns]
if len(standard_columns) != len(set(standard_columns)):
    collisions = pd.Series(standard_columns)[pd.Series(standard_columns).duplicated(keep=False)].tolist()
    raise ValueError(
        "Column-name standardization would create duplicate names: "
        f"{collisions}. Resolve these manually before proceeding."
    )

column_name_mapping = pd.DataFrame({
    "original_column_name": [str(c) for c in original_columns],
    "standardized_column_name": standard_columns
})

df = df_raw.copy(deep=True)
df.columns = standard_columns
display(column_name_mapping)

def propose_business_meaning(column_name):
    name = str(column_name).lower()
    rules = [
        (r"order_id|order_number|invoice_id", "Order/invoice identifier; one order may have multiple line items."),
        (r"customer_id|client_id|user_id", "Customer/user identifier; not a numeric measurement."),
        (r"product_id|sku|item_id", "Product/item identifier or stock-keeping code."),
        (r"product_name|item_name|description", "Product or item description."),
        (r"categor|department|segment", "Product/customer/business category or segment."),
        (r"quantity|qty|units", "Number of units ordered or sold; negative values may represent returns in some systems."),
        (r"unit_price|price_per_unit|unit_cost|^price$|selling_price", "Price per unit; confirm currency and discount/tax treatment."),
        (r"total|amount|revenue|sales|subtotal", "Monetary amount; confirm whether line-level or order-level."),
        (r"discount|coupon", "Discount amount or rate; confirm representation."),
        (r"tax|vat|gst", "Tax amount or rate."),
        (r"date|time|timestamp|created|shipped|delivered", "Order, shipment, delivery, or other event date/time."),
        (r"status|state", "Order, payment, shipment, or record status."),
        (r"city|region|country|postal|zip|address", "Geographic or delivery-location attribute."),
        (r"payment|method|shipping|ship_mode", "Payment or shipping method/type."),
        (r"email|phone|name", "Customer/contact information; apply privacy safeguards.")
    ]
    for pattern, meaning in rules:
        if re.search(pattern, name):
            return meaning
    return "Confirm meaning from the dataset documentation or data owner."

def infer_field_type(col, series):
    name = str(col).lower()
    if re.search(r"(^|_)(id|code|sku|number)(_|$)", name) or name.endswith("_id"):
        return "Identifier"
    if re.search(r"date|time|timestamp", name):
        return "Date/time candidate"
    if re.search(r"quantity|qty|units|price|amount|total|revenue|sales|discount|tax|cost", name):
        return "Numeric candidate"
    if pd.api.types.is_numeric_dtype(series):
        return "Numeric"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "Date/time"
    return "Categorical/text candidate"

business_dictionary = pd.DataFrame({
    "column": df.columns.astype(str),
    "loaded_dtype": [str(df[c].dtype) for c in df.columns],
    "proposed_field_type": [infer_field_type(c, df[c]) for c in df.columns],
    "proposed_business_meaning": [propose_business_meaning(c) for c in df.columns],
    "needs_owner_confirmation": True
})
display(business_dictionary)


,original_column_name,standardized_column_name
0,Order ID,order_id
1,Order Date,order_date
2,Customer ID,customer_id
3,Customer Name,customer_name
4,Product ID,product_id
5,Product Name,product_name
6,Category,category
7,Subcategory,subcategory
8,Quantity,quantity
9,Unit Price,unit_price


,column,loaded_dtype,proposed_field_type,proposed_business_meaning,needs_owner_confirmation
0,order_id,object,Identifier,Order/invoice identifier; one order may have m...,True
1,order_date,object,Date/time candidate,"Order, shipment, delivery, or other event date...",True
2,customer_id,object,Identifier,Customer/user identifier; not a numeric measur...,True
3,customer_name,object,Categorical/text candidate,Customer/contact information; apply privacy sa...,True
4,product_id,object,Identifier,Product/item identifier or stock-keeping code.,True
5,product_name,object,Categorical/text candidate,Product or item description.,True
6,category,object,Categorical/text candidate,Product/customer/business category or segment.,True
7,subcategory,object,Categorical/text candidate,Product/customer/business category or segment.,True
8,quantity,float64,Numeric candidate,Number of units ordered or sold; negative valu...,True
9,unit_price,float64,Numeric candidate,Price per unit; confirm currency and discount/...,True


## 4. Missing-value report and documented handling decisions

Percentages use the total number of rows as denominator. Default policy: do not impute important transaction fields without a defensible business rule. For descriptive categorical fields, `Unknown` may be appropriate, but it is not applied automatically because the exact fields and business semantics need confirmation.

In [4]:
def missing_report(frame):
    n = len(frame)
    return pd.DataFrame({
        "column": frame.columns.astype(str),
        "missing_count": frame.isna().sum().values,
        "missing_percent": (frame.isna().mean().values * 100).round(2)
    }).sort_values(["missing_count", "column"], ascending=[False, True])

missing_before = missing_report(df)
display(missing_before)
blank_rows_before = int(df.isna().all(axis=1).sum())
print("Entirely blank rows:", blank_rows_before)

def missing_decision(col):
    name = str(col).lower()
    if re.search(r"order_id|customer_id|product_id|_id$", name):
        return "Flag missing identifiers; do not invent IDs or drop without impact review."
    if re.search(r"quantity|qty|unit_price|price|total|amount|revenue", name):
        return "Flag for investigation; do not impute transactional values without a documented rule."
    if re.search(r"date|time|timestamp", name):
        return "Flag missing date/time; infer only from a verified source or business rule."
    if re.search(r"categor|status|payment|ship_mode|region|city|country", name):
        return "Consider an explicit 'Unknown' label only if valid for this field; review before applying."
    return "Review business importance; retain missing value unless a justified handling rule is approved."

missing_decisions = missing_before.copy()
missing_decisions["handling_decision"] = [
    "No missing values; no action." if count == 0 else missing_decision(col)
    for col, count in zip(missing_decisions["column"], missing_decisions["missing_count"])
]
display(missing_decisions)


,column,missing_count,missing_percent
13,delivery_status,181,1.80
12,payment_mode,180,1.79
9,unit_price,121,1.21
6,category,120,1.20
14,city,120,1.20
3,customer_name,119,1.19
5,product_name,79,0.79
8,quantity,79,0.79
1,order_date,62,0.62
2,customer_id,59,0.59


Entirely blank rows: 0


,column,missing_count,missing_percent,handling_decision
13,delivery_status,181,1.80,Consider an explicit 'Unknown' label only if v...
12,payment_mode,180,1.79,Consider an explicit 'Unknown' label only if v...
9,unit_price,121,1.21,Flag for investigation; do not impute transact...
6,category,120,1.20,Consider an explicit 'Unknown' label only if v...
14,city,120,1.20,Consider an explicit 'Unknown' label only if v...
3,customer_name,119,1.19,Review business importance; retain missing val...
5,product_name,79,0.79,Review business importance; retain missing val...
8,quantity,79,0.79,Flag for investigation; do not impute transact...
1,order_date,62,0.62,Flag missing date/time; infer only from a veri...
2,customer_id,59,0.59,Flag missing identifiers; do not invent IDs or...


## 5. Exact duplicates and repeated order IDs

Fully identical rows and repeated order identifiers are separate checks. A repeated order ID may be a valid order with multiple products. The notebook removes only exact duplicate rows after standardization and keeps a copy of removed rows in the audit output.

In [5]:
exact_duplicate_mask_raw = df.duplicated(keep="first")
exact_duplicates_raw = df.loc[exact_duplicate_mask_raw].copy()
print("Exact duplicate rows in standardized raw data (beyond first occurrence):", int(exact_duplicate_mask_raw.sum()))

order_id_candidates = [c for c in df.columns if re.search(r"(^|_)order(_)?id$|order_number|invoice_id", c, re.I)]
if order_id_candidates:
    order_id_col = order_id_candidates[0]
    nonblank_ids = df.loc[df[order_id_col].notna(), order_id_col].astype(str)
    repeated_order_id_counts = nonblank_ids.value_counts()
    repeated_order_id_counts = repeated_order_id_counts[repeated_order_id_counts > 1]
    repeated_order_ids = repeated_order_id_counts.rename_axis("order_id").reset_index(name="row_count")
    print("Order ID column used:", order_id_col)
    print("Order IDs appearing more than once:", len(repeated_order_ids))
    display(repeated_order_ids.head(30))

    # Show rows for repeated IDs to support investigation.
    repeated_id_values = set(repeated_order_ids["order_id"].astype(str))
    repeated_order_rows = df.loc[
        df[order_id_col].notna() & df[order_id_col].astype(str).isin(repeated_id_values)
    ].sort_values(order_id_col)
    display(repeated_order_rows.head(50))
else:
    order_id_col = None
    repeated_order_ids = pd.DataFrame(columns=["order_id", "row_count"])
    repeated_order_rows = pd.DataFrame()
    print("No likely order ID column found. Inspect the column dictionary and configure manually.")

print("Interpretation: repeated IDs are investigated, not removed automatically.")


Exact duplicate rows in standardized raw data (beyond first occurrence): 25
Order ID column used: order_id
Order IDs appearing more than once: 2487


,order_id,row_count
0,CA-2017-100111,14
1,CA-2017-157987,12
2,US-2016-108504,11
3,CA-2016-165330,11
4,US-2015-126977,10
5,CA-2015-131338,10
6,CA-2016-105732,10
7,US-2016-114013,9
8,CA-2017-117457,9
9,CA-2015-132626,9


,order_id,order_date,customer_id,customer_name,product_id,product_name,category,subcategory,quantity,unit_price,total_amount,discount,payment_mode,delivery_status,city,region
9806,CA-2014-100090,2014-07-08 00:00:00,EB-13705,Ed Braxton,FUR-TA-10003715,Hon 2111 Invitation Series Corner Table,Furniture,Tables,3.0,167.50,502.49,0.20,Net Banking,Delivered,San Francisco,West
440,CA-2014-100090,2014-07-08 00:00:00,EB-13705,Ed Braxton,OFF-BI-10001597,"Wilson Jones Ledger-Size, Piano-Hinge Binder, ...",Office Supplies,Binders,6.0,32.78,196.70,0.20,UPI,Delivered,San Francisco,West
4707,CA-2014-100363,2014-04-08 00:00:00,JM-15655,Jim Mitchum,OFF-PA-10004733,Things To Do Today Spiral Book,Office Supplies,Paper,3.0,6.34,19.01,0.20,Debit Card,Delivered,Glendale,West
9378,CA-2014-100363,2014-04-08 00:00:00,JM-15655,Jim Mitchum,OFF-FA-10000611,Binder Clips by OIC,Office Supplies,Fasteners,2.0,1.18,2.37,0.20,Cash on Delivery,Delivered,Glendale,West
8459,CA-2014-100678,2014-04-18 00:00:00,KM-16720,Kunst Miller,OFF-AR-10001868,Prang Dustless Chalk Sticks,Office Supplies,Art,2.0,1.34,2.69,0.20,Cash on Delivery,Delivered,Houston,Central
7046,CA-2014-100678,2014-04-18 00:00:00,KM-16720,Kunst Miller,TEC-AC-10000474,Kensington Expert Mouse Optical USB Trackball ...,Technology,Accessories,3.0,75.99,227.98,0.20,Debit Card,Shipped,Houston,Central
7931,CA-2014-100678,2014-04-18 00:00:00,KM-16720,Kunst Miller,OFF-EN-10000056,Cameo Buff Policy Envelopes,Office Supplies,Envelopes,3.0,49.78,149.35,0.20,UPI,Delivered,Houston,Central
1814,CA-2014-100678,2014-04-18 00:00:00,KM-16720,Kunst Miller,FUR-CH-10002602,DMI Arturo Collection Mission-style Design Woo...,Furniture,Chairs,3.0,105.69,317.06,0.30,Credit Card,Shipped,Houston,Central
9483,CA-2014-100706,2014-12-16 00:00:00,LE-16810,Laurel Elliston,TEC-AC-10001314,Case Logic 2.4GHz Wireless Keyboard,Technology,Accessories,2.0,49.99,99.98,0.00,Net Banking,Delivered,Springfield,South
5367,CA-2014-100706,2014-12-16 00:00:00,LE-16810,Laurel Elliston,FUR-FU-10002268,Ultra Door Push Plate,Furniture,Furnishings,6.0,4.91,29.46,0.00,Credit Card,Shipped,Springfield,South


Interpretation: repeated IDs are investigated, not removed automatically.


## 6. Standardize text and review category variants

Whitespace is normalized across text columns. For likely category/status/location columns, case is normalized to title case, excluding identifiers and code-like fields. This does not automatically fix spelling errors; frequency reports help identify variants for an explicit, reviewed mapping.

In [6]:
df_clean = df.copy(deep=True)
text_columns = df_clean.select_dtypes(include=["object", "string"]).columns.tolist()
category_pattern = re.compile(r"categor|segment|status|payment|ship_mode|country|region|city", re.I)
identifier_pattern = re.compile(r"(^|_)(id|sku|code|barcode|postal|zip|phone|email)(_|$)", re.I)

text_changes = []
for col in text_columns:
    before = df_clean[col].copy()
    after = before.map(lambda x: re.sub(r"\s+", " ", x.strip()) if isinstance(x, str) else x)
    apply_title = bool(category_pattern.search(col) and not identifier_pattern.search(col))
    if apply_title:
        after = after.map(lambda x: x.title() if isinstance(x, str) else x)
    df_clean[col] = after
    changed = before.astype("string").fillna("<NA>") != after.astype("string").fillna("<NA>")
    text_changes.append({
        "column": col,
        "changed_values": int(changed.sum()),
        "title_case_applied": apply_title
    })
text_standardization_report = pd.DataFrame(text_changes)
display(text_standardization_report)

category_columns = [c for c in df_clean.columns if category_pattern.search(c) and not identifier_pattern.search(c)]
category_value_reports = []
for col in category_columns:
    counts = df_clean[col].value_counts(dropna=False).rename_axis("value").reset_index(name="count")
    counts.insert(0, "column", col)
    category_value_reports.append(counts)
category_value_report = pd.concat(category_value_reports, ignore_index=True) if category_value_reports else pd.DataFrame(columns=["column", "value", "count"])
display(category_value_report.head(100))

# Optional explicit correction map. Populate only after reviewing value frequencies and confirming intended equivalences.
# Example: category_mapping = {"Electornics": "Electronics"}
category_mapping = {}
category_mapping_report = []
for col in category_columns:
    before = df_clean[col].copy()
    df_clean[col] = df_clean[col].replace(category_mapping)
    category_mapping_report.append({
        "column": col,
        "explicit_mapping_applied": bool(category_mapping),
        "values_changed": int((before.astype("string").fillna("<NA>") != df_clean[col].astype("string").fillna("<NA>")).sum())
    })
category_mapping_report = pd.DataFrame(category_mapping_report)


,column,changed_values,title_case_applied
0,order_id,0,False
1,order_date,0,False
2,customer_id,0,False
3,customer_name,0,False
4,product_id,0,False
5,product_name,273,False
6,category,55,True
7,subcategory,0,True
8,payment_mode,4294,True
9,delivery_status,54,True


,column,value,count
0,category,Office Supplies,5954
1,category,Furniture,2115
2,category,Technology,1845
3,category,NaN,120
4,subcategory,Binders,1529
...,...,...,...
95,city,Mesa,27
96,city,Fresno,26
97,city,Tulsa,26
98,city,Roseville,26


## 7. Numeric validation

Likely quantity, price, and amount columns are detected by their names. Numeric-looking text is converted where possible; nonblank values that cannot be converted are saved in a validation report with their original values. Negative quantities and zero/negative prices are flagged, not automatically deleted: returns, free products, and corrections may be legitimate.

In [7]:
def find_columns(pattern, columns):
    return [c for c in columns if re.search(pattern, str(c), re.I)]

quantity_cols = find_columns(r"quantity|qty|units", df_clean.columns)
unit_price_cols = find_columns(r"unit_price|price_per_unit|unit_cost|^price$|selling_price", df_clean.columns)
total_cols = find_columns(r"(^total$|total_amount|line_total|subtotal|sales_amount|revenue|amount)", df_clean.columns)

numeric_validation_rows = []
numeric_parse_issues = []
for col in sorted(set(quantity_cols + unit_price_cols + total_cols)):
    raw = df_clean[col].copy()
    converted = pd.to_numeric(raw, errors="coerce")
    invalid = raw.notna() & converted.isna()
    for idx in raw.index[invalid]:
        numeric_parse_issues.append({"source_row_index": idx, "column": col, "raw_value": str(raw.loc[idx])})
    role = "quantity" if col in quantity_cols else "unit_price" if col in unit_price_cols else "total/amount"
    numeric_validation_rows.append({
        "column": col,
        "role": role,
        "nonblank_before_conversion": int(raw.notna().sum()),
        "unparseable_nonblank": int(invalid.sum()),
        "negative_values": int((converted < 0).sum()),
        "zero_values": int((converted == 0).sum()),
        "missing_after_conversion": int(converted.isna().sum())
    })
    df_clean[col] = converted

numeric_validation_report = pd.DataFrame(numeric_validation_rows)
numeric_parse_issues_report = pd.DataFrame(numeric_parse_issues, columns=["source_row_index", "column", "raw_value"])
display(numeric_validation_report)
display(numeric_parse_issues_report.head(50))

negative_quantity_report = df_clean.loc[df_clean[quantity_cols].lt(0).any(axis=1)].copy() if quantity_cols else pd.DataFrame()
nonpositive_price_report = df_clean.loc[df_clean[unit_price_cols].le(0).any(axis=1)].copy() if unit_price_cols else pd.DataFrame()
print("Rows with negative quantities:", len(negative_quantity_report))
print("Rows with zero or negative unit prices:", len(nonpositive_price_report))


,column,role,nonblank_before_conversion,unparseable_nonblank,negative_values,zero_values,missing_after_conversion
0,quantity,quantity,9955,0,14,10,79
1,total_amount,total/amount,10034,0,0,0,0
2,unit_price,unit_price,9913,0,12,8,121


,source_row_index,column,raw_value


Rows with negative quantities: 14
Rows with zero or negative unit prices: 20


## 8. Date validation

Likely date columns are parsed and invalid nonblank values are captured with their original raw values before conversion. If dates are ambiguous (such as `03/04/2025`), verify the source convention and use an explicit format before finalizing.

In [8]:
date_cols = find_columns(r"date|datetime|timestamp|created_at|updated_at|shipped_at|delivered_at", df_clean.columns)
date_validation_rows = []
invalid_date_details = []

for col in date_cols:
    raw = df_clean[col].copy()
    parsed = pd.to_datetime(raw, errors="coerce")
    invalid = raw.notna() & parsed.isna()
    date_validation_rows.append({
        "column": col,
        "nonblank_before_parse": int(raw.notna().sum()),
        "invalid_nonblank_dates": int(invalid.sum()),
        "earliest_valid_date": parsed.min(),
        "latest_valid_date": parsed.max()
    })
    for idx in raw.index[invalid]:
        invalid_date_details.append({
            "source_row_index": idx,
            "column": col,
            "raw_value": str(raw.loc[idx])
        })
    df_clean[col] = parsed

date_validation_report = pd.DataFrame(date_validation_rows)
invalid_dates_report = pd.DataFrame(invalid_date_details, columns=["source_row_index", "column", "raw_value"])
display(date_validation_report)
display(invalid_dates_report.head(50))


,column,nonblank_before_parse,invalid_nonblank_dates,earliest_valid_date,latest_valid_date
0,order_date,9972,5,2014-01-03,2035-01-15


,source_row_index,column,raw_value
0,2572,order_date,32/13/2022
1,5212,order_date,not a date
2,6607,order_date,2023-15-40
3,7356,order_date,32/13/2022
4,9462,order_date,not a date


## 9. Total consistency checks

If quantity, unit price, and a total/amount column are available, compare `quantity × unit price` with the observed amount. This is a diagnostic only. Discounts, taxes, shipping, returns, currency conversion, and order-level totals may explain differences. Confirm that the chosen amount is a line total before interpreting mismatches.

In [9]:
# Review and override these mappings if multiple candidate columns exist.
quantity_col = quantity_cols[0] if quantity_cols else None
price_col = unit_price_cols[0] if unit_price_cols else None
total_col = total_cols[0] if total_cols else None

ABS_TOLERANCE = 0.02
REL_TOLERANCE = 0.01

if quantity_col and price_col and total_col:
    q = pd.to_numeric(df_clean[quantity_col], errors="coerce")
    p = pd.to_numeric(df_clean[price_col], errors="coerce")
    observed = pd.to_numeric(df_clean[total_col], errors="coerce")
    expected = q * p
    applicable = q.notna() & p.notna() & observed.notna()
    difference = observed - expected
    tolerance = ABS_TOLERANCE + REL_TOLERANCE * expected.abs()
    total_consistency_report = pd.DataFrame({
        "source_row_index": df_clean.index,
        "quantity": q,
        "unit_price": p,
        "observed_total": observed,
        "expected_quantity_x_price": expected,
        "difference": difference,
        "check_applicable": applicable,
        "mismatch_flag": applicable & (difference.abs() > tolerance)
    })
    print("Columns used:", quantity_col, price_col, total_col)
    print("Applicable comparisons:", int(applicable.sum()))
    print("Potential mismatches:", int(total_consistency_report["mismatch_flag"].sum()))
    display(total_consistency_report.loc[total_consistency_report["mismatch_flag"]].head(50))
else:
    total_consistency_report = pd.DataFrame()
    print("Skipped: suitable quantity, unit-price, and total columns were not all detected.")


Columns used: quantity unit_price total_amount
Applicable comparisons: 9835
Potential mismatches: 87


,source_row_index,quantity,unit_price,observed_total,expected_quantity_x_price,difference,check_applicable,mismatch_flag
337,337,-5.0,62.18,62.18,-310.90,373.08,True,True
343,343,4.0,19.35,63.86,77.40,-13.54,True,True
395,395,3.0,0.00,44.94,0.00,44.94,True,True
473,473,4.0,3.86,19.27,15.44,3.83,True,True
532,532,4.0,0.00,9.22,0.00,9.22,True,True
685,685,3.0,179.99,269.99,539.97,-269.98,True,True
715,715,4.0,4.78,15.79,19.12,-3.33,True,True
795,795,9.0,0.00,38.52,0.00,38.52,True,True
816,816,-5.0,7.04,35.20,-35.20,70.40,True,True
841,841,7.0,-1.99,6.29,-13.93,20.22,True,True


## 10. Apply conservative cleaning rules

Automatic changes are limited to:
1. Standardized column names and whitespace/case normalization.
2. Numeric/date type conversion for detected fields, with invalid original values recorded in audit reports.
3. Removal of entirely blank rows.
4. Removal of fully identical rows after normalization, keeping the first occurrence.

Other anomalies are retained and reported for review. This avoids irreversible deletion of potentially meaningful transactions.

In [10]:
rows_before_rules = len(df_clean)
blank_row_mask = df_clean.isna().all(axis=1)
blank_rows_removed = int(blank_row_mask.sum())
blank_rows_audit = df_clean.loc[blank_row_mask].copy()
df_clean = df_clean.loc[~blank_row_mask].copy()

normalized_duplicate_mask = df_clean.duplicated(keep="first")
normalized_duplicates_removed = df_clean.loc[normalized_duplicate_mask].copy()
normalized_duplicate_count = int(normalized_duplicate_mask.sum())
df_clean = df_clean.loc[~normalized_duplicate_mask].copy()

print("Rows before removal rules:", rows_before_rules)
print("Blank rows removed:", blank_rows_removed)
print("Exact duplicates removed after normalization:", normalized_duplicate_count)
print("Rows in cleaned dataset:", len(df_clean))
print("Repeated order IDs retained unless they were exact duplicate rows.")


Rows before removal rules: 10034
Blank rows removed: 0
Exact duplicates removed after normalization: 25
Rows in cleaned dataset: 10009
Repeated order IDs retained unless they were exact duplicate rows.


## 11. Before-and-after quality summary and missingness

The before summary uses the loaded dataset with standardized column labels for consistency. The after summary uses the cleaned dataset. Invalid nonblank numeric/date values converted to missing are separately recorded in their respective reports.

In [11]:
def quality_summary(frame, stage):
    total_cells = frame.shape[0] * frame.shape[1]
    return {
        "stage": stage,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
        "missing_percent_of_cells": round(frame.isna().sum().sum() / total_cells * 100, 2) if total_cells else 0,
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "entirely_blank_rows": int(frame.isna().all(axis=1).sum())
    }

quality_comparison = pd.DataFrame([
    quality_summary(df, "Before cleaning"),
    quality_summary(df_clean, "After cleaning")
])
missing_after = missing_report(df_clean)
display(quality_comparison)
display(missing_after)


,stage,rows,columns,missing_cells,missing_percent_of_cells,exact_duplicate_rows,entirely_blank_rows
0,Before cleaning,10034,16,1120,0.7,25,0
1,After cleaning,10009,16,1121,0.7,0,0


,column,missing_count,missing_percent
13,delivery_status,179,1.79
12,payment_mode,179,1.79
6,category,120,1.20
14,city,120,1.20
9,unit_price,120,1.20
3,customer_name,119,1.19
5,product_name,79,0.79
8,quantity,79,0.79
1,order_date,67,0.67
2,customer_id,59,0.59


## 12. Save cleaned dataset and audit files

Outputs are stored in a separate project folder. The raw source workbook is not overwritten.

In [12]:
audit_log = pd.DataFrame([
    {"check": "Source file", "result": file_path, "decision": "Read only; original preserved"},
    {"check": "Original rows", "result": len(df_raw), "decision": "Baseline"},
    {"check": "Original columns", "result": df_raw.shape[1], "decision": "Baseline"},
    {"check": "Entirely blank rows removed", "result": blank_rows_removed, "decision": "Removed and audited"},
    {"check": "Exact duplicates removed after normalization", "result": normalized_duplicate_count, "decision": "Kept first; removed copies audited"},
    {"check": "Repeated order IDs", "result": len(repeated_order_ids), "decision": "Investigated and retained unless exact duplicate"},
    {"check": "Missing cells after cleaning", "result": int(df_clean.isna().sum().sum()), "decision": "Not imputed without justified business rule"},
    {"check": "Invalid numeric values", "result": len(numeric_parse_issues_report), "decision": "Original raw values saved for review"},
    {"check": "Invalid date values", "result": len(invalid_dates_report), "decision": "Original raw values saved for review"},
    {"check": "Total mismatches", "result": int(total_consistency_report["mismatch_flag"].sum()) if len(total_consistency_report) and "mismatch_flag" in total_consistency_report else "Not checked", "decision": "Flagged only; business context required"}
])

cleaned_xlsx = data_cleaned_dir / "ecommerce_cleaned_dataset.xlsx"
audit_xlsx = reports_dir / "ecommerce_data_quality_audit.xlsx"

with pd.ExcelWriter(cleaned_xlsx, engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="Cleaned_Data", index=False)

with pd.ExcelWriter(audit_xlsx, engine="openpyxl") as writer:
    quality_comparison.to_excel(writer, sheet_name="Quality_Summary", index=False)
    missing_before.to_excel(writer, sheet_name="Missing_Before", index=False)
    missing_after.to_excel(writer, sheet_name="Missing_After", index=False)
    missing_decisions.to_excel(writer, sheet_name="Missing_Decisions", index=False)
    column_name_mapping.to_excel(writer, sheet_name="Column_Name_Mapping", index=False)
    business_dictionary.to_excel(writer, sheet_name="Business_Dictionary", index=False)
    audit_log.to_excel(writer, sheet_name="Audit_Log", index=False)
    text_standardization_report.to_excel(writer, sheet_name="Text_Standardization", index=False)
    category_value_report.to_excel(writer, sheet_name="Category_Values", index=False)
    category_mapping_report.to_excel(writer, sheet_name="Category_Mapping", index=False)
    numeric_validation_report.to_excel(writer, sheet_name="Numeric_Validation", index=False)
    numeric_parse_issues_report.to_excel(writer, sheet_name="Numeric_Parse_Issues", index=False)
    date_validation_report.to_excel(writer, sheet_name="Date_Validation", index=False)
    invalid_dates_report.to_excel(writer, sheet_name="Invalid_Dates", index=False)
    repeated_order_ids.to_excel(writer, sheet_name="Repeated_Order_IDs", index=False)
    repeated_order_rows.to_excel(writer, sheet_name="Repeated_Order_Rows", index=False)
    exact_duplicates_raw.to_excel(writer, sheet_name="Raw_Exact_Duplicates", index=False)
    normalized_duplicates_removed.to_excel(writer, sheet_name="Removed_Duplicates", index=False)
    blank_rows_audit.to_excel(writer, sheet_name="Blank_Rows_Removed", index=False)
    negative_quantity_report.to_excel(writer, sheet_name="Negative_Quantities", index=False)
    nonpositive_price_report.to_excel(writer, sheet_name="Nonpositive_Prices", index=False)
    total_consistency_report.to_excel(writer, sheet_name="Total_Consistency", index=False)

csv_reports = {
    "quality_comparison.csv": quality_comparison,
    "missing_values_before.csv": missing_before,
    "missing_values_after.csv": missing_after,
    "missing_value_decisions.csv": missing_decisions,
    "repeated_order_ids.csv": repeated_order_ids,
    "numeric_validation.csv": numeric_validation_report,
    "numeric_parse_issues.csv": numeric_parse_issues_report,
    "date_validation.csv": date_validation_report,
    "invalid_dates.csv": invalid_dates_report,
    "total_consistency.csv": total_consistency_report,
    "audit_log.csv": audit_log
}
for filename, report in csv_reports.items():
    report.to_csv(reports_dir / filename, index=False)

print("Cleaned dataset:", cleaned_xlsx)
print("Audit workbook:", audit_xlsx)
print("Reports folder:", reports_dir)


Cleaned dataset: /content/drive/MyDrive/data-science-week1-assignment/data/cleaned/ecommerce_cleaned_dataset.xlsx
Audit workbook: /content/drive/MyDrive/data-science-week1-assignment/reports/ecommerce_data_quality_audit.xlsx
Reports folder: /content/drive/MyDrive/data-science-week1-assignment/reports


## 13. Conclusion

The following conclusion is generated from the actual run. Interpret flagged anomalies in the context of the business rules and confirm any assumptions before using the dataset for further analysis.

In [13]:
print("FINAL CONCLUSION")
print("=" * 70)
print(f"Raw dataset: {len(df_raw):,} rows × {df_raw.shape[1]:,} columns.")
print(f"Cleaned dataset: {len(df_clean):,} rows × {df_clean.shape[1]:,} columns.")
print(f"Entirely blank rows removed: {blank_rows_removed:,}.")
print(f"Exact duplicate rows removed after normalization: {normalized_duplicate_count:,}.")
print(f"Missing cells remaining: {int(df_clean.isna().sum().sum()):,}.")
print(f"Repeated order IDs flagged for review: {len(repeated_order_ids):,}.")
print(f"Unparseable nonblank numeric values: {len(numeric_parse_issues_report):,}.")
print(f"Invalid nonblank dates: {len(invalid_dates_report):,}.")
if len(total_consistency_report) and "mismatch_flag" in total_consistency_report:
    print(f"Potential total mismatches: {int(total_consistency_report['mismatch_flag'].sum()):,}.")
else:
    print("Total consistency check was not applicable based on detected columns.")
print("\nOnly blank rows and confirmed exact duplicate rows were automatically removed.")
print("Missing business fields and suspicious values were retained and documented for review.")
print("The raw source file was not overwritten.")
print(f"Cleaned dataset and reports are saved under: {project_dir}")


FINAL CONCLUSION
Raw dataset: 10,034 rows × 16 columns.
Cleaned dataset: 10,009 rows × 16 columns.
Entirely blank rows removed: 0.
Exact duplicate rows removed after normalization: 25.
Missing cells remaining: 1,121.
Repeated order IDs flagged for review: 2,487.
Unparseable nonblank numeric values: 0.
Invalid nonblank dates: 5.
Potential total mismatches: 87.

Only blank rows and confirmed exact duplicate rows were automatically removed.
Missing business fields and suspicious values were retained and documented for review.
The raw source file was not overwritten.
Cleaned dataset and reports are saved under: /content/drive/MyDrive/data-science-week1-assignment
